### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd

In [2]:
# from unsloth import FastLanguageModel
# import torch

# max_seq_length = 2048 
# dtype = ( None )
# load_in_4bit = False 


# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="./lora/lora_16bit_merged/",
#     max_seq_length=max_seq_length,
#     dtype=dtype,
#     load_in_4bit=load_in_4bit,
# )
# FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [3]:
# # I highly do NOT suggest - use Unsloth if possible
# from peft import AutoPeftModelForCausalLM
# from transformers import AutoTokenizer
# model = AutoPeftModelForCausalLM.from_pretrained(
#     "lora_model", # YOUR MODEL YOU USED FOR TRAINING
#     load_in_4bit = False,
# )
# tokenizer = AutoTokenizer.from_pretrained("lora_model")

In [4]:
from transformers import AutoProcessor, AutoModel
import torch
from PIL import Image
import cairosvg
import os

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('DEVICE', DEVICE)

model_sl = AutoModel.from_pretrained("google/siglip-so400m-patch14-384").to(DEVICE)
processor_sl = AutoProcessor.from_pretrained("google/siglip-so400m-patch14-384")

def svgMetric(prompt, svg):
    try:
        # Convert SVG to PNG
        cairosvg.svg2png(svg, write_to="./tmp/temp.png")
        
        # Open and process the image
        image = Image.open('./tmp/temp.png').convert("RGB")
        texts = ["SVG illustration of " + prompt]
        inputs = processor_sl(text=texts, images=image, padding="max_length", return_tensors="pt").to(DEVICE)
        
        # Inference without gradient tracking
        with torch.no_grad():
            outputs = model_sl(**inputs)
        
        logits_per_image = outputs.logits_per_image
        probs = torch.sigmoid(logits_per_image)
        
        # Clean up temporary PNG file
        #os.remove('./tmp/temp.png')
        
        return probs[0][0].item()

    
    except Exception as e:
        print(f"An error occurred: {e}")
        return None


DEVICE cuda


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [5]:
import concurrent
import io
import logging
import re
import re2

import cairosvg
import kagglehub
import torch
from lxml import etree
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

svg_constraints = kagglehub.package_import('metric/svg-constraints')
# ###/home/vino/.cache/kagglehub/notebooks/metric/svg-constraints/output/versions/1
# svg_metrics = kagglehub.package_import('jiazhuang/svg-image-fidelity/versions/12')
# ###/home/vino/.cache/kagglehub/notebooks/jiazhuang/svg-image-fidelity/output/versions/12

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('DEVICE', DEVICE)

class SVGSanitizer:
    def __init__(self, constraints, default_svg):
        self.constraints = constraints
        self.default_svg = default_svg
    
    def enforce_constraints(self, svg_string: str) -> str:
        """Enforces constraints on an SVG string, removing disallowed elements
        and attributes.

        Parameters
        ----------
        svg_string : str
            The SVG string to process.

        Returns
        -------
        str
            The processed SVG string, or the default SVG if constraints
            cannot be satisfied.
        """
        logging.info('Sanitizing SVG...')

        try:
            parser = etree.XMLParser(remove_blank_text=True, remove_comments=True)
            root = etree.fromstring(svg_string, parser=parser)
        except etree.ParseError as e:
            logging.error('SVG Parse Error: %s. Returning default SVG.', e)
            return self.default_svg
    
        elements_to_remove = []
        for element in root.iter():
            tag_name = etree.QName(element.tag).localname
    
            # Remove disallowed elements
            if tag_name not in self.constraints.allowed_elements:
                elements_to_remove.append(element)
                continue  # Skip attribute checks for removed elements
    
            # Remove disallowed attributes
            attrs_to_remove = []
            for attr in element.attrib:
                attr_name = etree.QName(attr).localname
                if (
                    attr_name
                    not in self.constraints.allowed_elements[tag_name]
                    and attr_name
                    not in self.constraints.allowed_elements['common']
                ):
                    attrs_to_remove.append(attr)
    
            for attr in attrs_to_remove:
                logging.debug(
                    'Attribute "%s" for element "%s" not allowed. Removing.',
                    attr,
                    tag_name,
                )
                del element.attrib[attr]
    
            # Check and remove invalid href attributes
            for attr, value in element.attrib.items():
                 if etree.QName(attr).localname == 'href' and not value.startswith('#'):
                    logging.debug(
                        'Removing invalid href attribute in element "%s".', tag_name
                    )
                    del element.attrib[attr]

            # Validate path elements to help ensure SVG conversion
            if tag_name == 'path':
                d_attribute = element.get('d')
                if not d_attribute:
                    logging.warning('Path element is missing "d" attribute. Removing path.')
                    elements_to_remove.append(element)
                    continue # Skip further checks for this removed element
                # Use regex to validate 'd' attribute format
                path_regex = re2.compile(
                    r'^'  # Start of string
                    r'(?:'  # Non-capturing group for each command + numbers block
                    r'[MmZzLlHhVvCcSsQqTtAa]'  # Valid SVG path commands (adjusted to exclude extra letters)
                    r'\s*'  # Optional whitespace after command
                    r'(?:'  # Non-capturing group for optional numbers
                    r'-?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?'  # First number
                    r'(?:[\s,]+-?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?)*'  # Subsequent numbers with mandatory separator(s)
                    r')?'  # Numbers are optional (e.g. for Z command)
                    r'\s*'  # Optional whitespace after numbers/command block
                    r')+'  # One or more command blocks
                    r'\s*'  # Optional trailing whitespace
                    r'$'  # End of string
                )
                if not path_regex.match(d_attribute):
                    logging.warning(
                        'Path element has malformed "d" attribute format. Removing path.'
                    )
                    elements_to_remove.append(element)
                    continue
                logging.debug('Path element "d" attribute validated (regex check).')
        
        # Remove elements marked for removal
        for element in elements_to_remove:
            if element.getparent() is not None:
                element.getparent().remove(element)
                logging.debug('Removed element: %s', element.tag)

        try:
            cleaned_svg_string = etree.tostring(root, encoding='unicode')
            return cleaned_svg_string
        except ValueError as e:
            logging.error(
                'SVG could not be sanitized to meet constraints: %s', e
            )
            return self.default_svg

class SVGProcessor:
    @staticmethod
    def clean_and_extract_svgs(text, default_svg):
        text = re.sub(r'^.*?(<svg\b)', r'\1', text, flags=re.DOTALL)
        svg_blocks = re.findall(r'<svg\b.*?</svg>', text, re.DOTALL)
    
        if svg_blocks:
            tmp = re.findall(r'<svg\b.*?', svg_blocks[-1], re.DOTALL)
            if len(tmp) > 1:
                tmp2 = svg_blocks[-1].split('<svg')
                return '<svg ' + tmp2[-1]
            else:
                return svg_blocks[-1]
        else:
            if "<svg" in text and "</svg>" not in text:
                text += "</svg>"
                return text
            return default_svg
    
    @staticmethod
    def svg_conversion_check(topic, base_svg_code, default_svg):
        try:
            cairosvg.svg2png(bytestring=base_svg_code.encode('utf-8'), write_to="temp.png")
            return base_svg_code
        except Exception as e:
            print(f"Failed to convert {topic} due to {str(e)}, Returning default SVG.")
            return default_svg


class Model:
    def __init__(self):
        
        ##Configure 4-bit quantization
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',  # Normalized float 4
            bnb_4bit_use_double_quant=False,  # Second quantization layer
            bnb_4bit_compute_dtype=torch.float16  # Computation in FP16
        )
        
        self.model_path = "./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1"
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_path)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_path,
            torch_dtype=torch.float16,
            #quantization_config=bnb_config,
            device_map="auto"
        )

        # Check model dtype and device
        for name, param in self.model.named_parameters():
            print(f"{name}: {param.dtype} on {param.device}")
            break  # remove break to list all parameters

        self.model.eval()
        
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)
    
    def get_response(self, description):

        instruction = """Generate SVG code to visually represent the following text description, while respecting the given constraints.
        <constraints>
        * **Allowed Elements:** `svg`, `path`, `circle`, `rect`, `ellipse`, `line`, `polyline`, `polygon`, `g`, `linearGradient`, `radialGradient`, `stop`, `defs`
        * **Allowed Attributes:** `viewBox`, `width`, `height`, `fill`, `stroke`, `stroke-width`, `d`, `cx`, `cy`, `r`, `x`, `y`, `rx`, `ry`, `x1`, `y1`, `x2`, `y2`, `points`, `transform`, `opacity`
        </constraints>
        
        <example>
        <description>"A red circle with a blue square inside"</description>
        ```svg
        <svg viewBox="0 0 256 256" width="256" height="256">
          <circle cx="50" cy="50" r="40" fill="red"/>
          <rect x="30" y="30" width="40" height="40" fill="blue"/>
        </svg>
        ```
        </example>
        
        
        Please ensure that the generated SVG code is well-formed, valid, and strictly adheres to these constraints.
        Focus on a clear and concise representation of the input description within the given limitations. 
        Always give the complete SVG code with nothing omitted. Never use an ellipsis.
        
        <description>"{}"</description>
        ```svg
        <svg viewBox="0 0 256 256" width="256" height="256">
        """
        
        alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
    
        ### Instruction:
        Please write an SVG code for the given input.
    
        ### Input:
        {}
    
        ### Response:
        """
        
        instruction= instruction.format(description)
        formatted_input = alpaca_prompt.format(instruction)
        #print(formatted_input)
        inputs = self.tokenizer([formatted_input], return_tensors="pt").to(DEVICE)
        outputs = self.model.generate(**inputs, temperature=0.5, max_new_tokens=2048, use_cache=True)
        return self.tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
    def predict(self, description: str, max_new_tokens=2048) -> str:
        output_decoded = self.get_response(description)
        base_svg_code = SVGProcessor.clean_and_extract_svgs(output_decoded, self.default_svg)
        clean_svg_code = self.sanitizer.enforce_constraints(base_svg_code)
        return SVGProcessor.svg_conversion_check(description, clean_svg_code, self.default_svg)



This code could modify your python environment or operating system.

Review this code at https://www.kaggle.com/code/metric/svg-constraints/versions/1
or in your download cache at /home/vino/.cache/kagglehub/notebooks/metric/svg-constraints/output/versions/1

It is strongly recommended that you run this code within a container
such as Docker to provide a secure, isolated execution environment.
See https://www.kaggle.com/docs/packages for more information.

Do you want to proceed? (y)es/[no]:  y


DEVICE cuda


In [6]:
model=Model()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

model.embed_tokens.weight: torch.float16 on cuda:0


In [7]:
model.predict('sun rising in the east')

'<svg viewBox="0 0 256 256" width="256" height="256"><defs><radialGradient id="sunGradient" cx="0.5" cy="0.5" r="0.5"><stop offset="0%" stop-color="yellow"/><stop offset="100%" stop-color="orange"/></radialGradient></defs><g transform="translate(128, 128)"><polyline points="128,32 130,50 130,68 135,60 140,65 145,55 150,55 155,50 160,60 165,45 170,55 175,50 180,60 185,45 190,55 195,55 200,50 205,60 210,45 215,55 220,50 225,60 230,45 235,55 240,50 245,60 250,45 255,55 260,50 265,60 270,45 275,55 280,50 285,60 290,45 295,55 300,50 305,60 310,45 315,55 320,50 325,60 330,45 335,55 340,50 345,60 350,45 355,55 360,50 365,60 370,45 375,55 380,50 385,60 390,45 395,55 400,50 405,60 410,45 415,55 420,50 425,60 430,45 435,55 440,50 445,60 450,45 455,55 460,50 465,60 470,45 475,55 480,50 485,60 490,45 495,55 500,50 505,60 510,45 515,55 520,50 525,60 530,45 535,55 540,50 545,60 550,45 555,55 560,50 565,60 570,45 575,55 580,50 585,60 590,45 595,55 600,50 605,60 610,45 615,55 620,50 625,60 630,45 635,

In [8]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/train.csv',header=[0])

In [9]:
from tqdm import tqdm
tqdm.pandas()
df['svg'] = df['description'].progress_apply(lambda x: model.predict(x))

100%|███████████████████████████████████████████| 15/15 [03:12<00:00, 12.81s/it]


In [15]:
#write csv file for new metric score
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
file_name = re.sub(r'[^a-zA-Z0-9]', '_', model.model_path)
df.to_csv(f'./io_files/pred_{file_name}_{timestamp}.csv', index=False)

In [11]:
#SigLip Score
df['sl_score'] = df.progress_apply(lambda row: svgMetric(row['description'], row['svg']), axis=1)


100%|███████████████████████████████████████████| 15/15 [00:01<00:00, 14.66it/s]


In [12]:
df['sl_score'].mean()

np.float64(0.3512861575436659)

In [13]:
df

,id,description,svg,sl_score
0,02d892,a purple forest at dusk,"<svg viewBox=""0 0 256 256"" width=""256"" height=...",5.179138e-01
1,0dcd2e,gray wool coat with a faux fur collar,"<svg viewBox=""0 0 256 256"" width=""256"" height=...",9.032997e-04
2,1e9ac1,a lighthouse overlooking the ocean,"<svg viewBox=""0 0 256 256"" width=""256"" height=...",1.664745e-02
3,2b25db,burgundy corduroy pants with patch pockets and...,"<svg viewBox=""0 0 256 256"" width=""256"" height=...",7.669778e-01
4,4e6a54,orange corduroy overalls,"<svg viewBox=""0 0 256 256"" width=""256"" height=...",7.445454e-06
5,4f1b00,a purple silk scarf with tassel trim,"<svg viewBox=""0 0 256 256"" width=""256"" height=...",1.724947e-02
6,61b500,a green lagoon under a cloudy sky,"<svg viewBox=""0 0 256 256"" width=""256"" height=...",9.634107e-01
7,65cc74,crimson rectangles forming a chaotic grid,"<svg viewBox=""0 0 256 256"" width=""256"" height=...",7.283612e-01
8,7c4414,purple pyramids spiraling around a bronze cone,"<svg viewBox=""0 0 256 256"" width=""256"" height=...",1.746304e-05
9,996c3a,magenta trapezoids layered on a transluscent s...,"<svg viewBox=""0 0 256 256"" width=""256"" height=...",6.526298e-01
